In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive
from IPython.display import display, HTML

# 1. MathJax & CSS
display(HTML(r"""
<script>
    if (window.MathJax) {
        MathJax.Hub.Config({
            "HTML-CSS": { scale: 130 },
            SVG: { scale: 130 }
        });
    }
</script>
<style>
    .rendered_html math { font-size: 1.3em !important; }
    .output_subarea, .jp-OutputArea-output { max-height: none !important; }
</style>
"""))

sp.init_printing(use_latex='mathjax')

# 2. Symbolic variables
omega, A = sp.symbols('omega A', real=True)
n, L = sp.symbols('n L', integer=True, positive=True)

# 3. Signal definition
x_n = A

# 4. DTFT directly from the definition
X_sum = sp.summation(
    x_n * sp.exp(-sp.I * omega * n),
    (n, 0, L - 1)
)

# 5. Algebraic simplification performed by SymPy
X_simplified = sp.simplify(X_sum)

# 6. Geometric-series form με χρήση trigsimp για την μορφή του βιβλίου
X_geometric = sp.trigsimp(X_simplified)

# 7. Verification
verification = sp.simplify(X_sum - X_geometric)

# 8. Real and imaginary parts
X_expanded = sp.expand_complex(X_geometric)
X_real = sp.simplify(sp.re(X_expanded))
X_imag = sp.simplify(sp.im(X_expanded))

# 9. Magnitude and Phase
mag_sym = sp.simplify(sp.Abs(X_geometric))
phase_sym = sp.simplify(sp.arg(X_geometric))

# 10. Display symbolic derivation
display(HTML(r"<h3 style='color:#2c3e50;'>Strict Symbolic DTFT Derivation (Proakis & Manolakis)</h3>"))
display(HTML(r"<b>1. DTFT from definition:</b>"))
display(sp.Eq(sp.Symbol(r'\mathcal{X}(e^{j\omega})'), X_sum))
display(HTML(r"<b>2. Simplified form (trigsimp):</b>"))
display(sp.Eq(sp.Symbol(r'\mathcal{X}(e^{j\omega})'), X_geometric))
display(HTML(r"<b>3. Magnitude:</b>"))
display(sp.Eq(sp.Symbol(r'|\mathcal{X}(e^{j\omega})|'), mag_sym))
display(HTML(r"<b>4. Phase:</b>"))
display(sp.Eq(sp.Symbol(r'\angle\mathcal{X}(e^{j\omega})'), phase_sym))

# 11. Interactive numerical spectra
def plot_rectangular_window_spectra(A_val, L_val):
    omega_vals = np.linspace(-2 * np.pi, 2 * np.pi, 1200)
    X_function = sp.lambdify((omega, A, L), X_geometric, modules='numpy')
    
    X_vals = np.empty(omega_vals.shape, dtype=complex)
    singular = np.isclose(np.sin(omega_vals / 2), 0, atol=1e-12)
    regular = ~singular
    
    with np.errstate(divide='ignore', invalid='ignore'):
        X_vals[regular] = X_function(omega_vals[regular], A_val, L_val)
        
    for idx in np.where(singular)[0]:
        X_vals[idx] = sum(A_val * np.exp(-1j * omega_vals[idx] * k) for k in range(L_val))
        
    amplitude = np.abs(X_vals)
    phase = np.angle(X_vals)
    phase[amplitude < 1e-10] = np.nan

    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    axes[0].plot(omega_vals/np.pi, amplitude, color='#1f77b4', lw=2)
    axes[0].set_title(f'Amplitude Spectrum (A={A_val:.3f}, L={L_val})')
    axes[0].grid(True, linestyle='--')
    axes[1].plot(omega_vals/np.pi, phase, color='#ff7f0e', lw=2)
    axes[1].set_title('Phase Spectrum')
    axes[1].set_xlabel(r'Frequency ($\omega/\pi$)')
    axes[1].grid(True, linestyle='--')
    plt.tight_layout()
    plt.show()

interactive_plot = interactive(
    plot_rectangular_window_spectra,
    A_val=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='A:'),
    L_val=widgets.IntSlider(value=5, min=2, max=25, step=1, description='L:')
)
display(interactive_plot)